# ML-агент: проверка разделения моделей по `BUSINESS_TYPE`

Агент выполняет задание из `tasks/business_type_split.txt`:

1. обучает исходную модель на всех данных (как в `main_fit`);
2. обучает отдельную модель на каждом значении `BUSINESS_TYPE` через `CascoFLAutoML(...).update_models()`;
3. если обучение падает — правит копию конфига (встроенные правила + LLM) и перезапускает;
4. сравнивает сегментную и исходную модели на одних и тех же тестовых строках сегмента;
5. отправляет результаты в LLM, получает выводы (JSON) и рассылает письмо.

Все настройки (пути, почта, LLM) — в `config.py`, постановка задачи — в txt.

## 1. Подготовка

In [ ]:
import sys
import importlib
from pathlib import Path

# Корень casco-проекта: config.py, casco_automl.py, model_configs/, temp_dataset_main_fit.gzip
PROJECT_DIR = Path.cwd()
# корень проекта первым в sys.path: импортируется config.py именно этого проекта
if str(PROJECT_DIR) in sys.path:
    sys.path.remove(str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR))

import config
importlib.reload(config)  # подхватить правки config.py без перезапуска ядра
print('config:', config.__file__, '| LLM:', config.llm_model_type)

from ml_agent import MLAgent, load_task, setup_logging

setup_logging('INFO')  # в ячейках — только сообщения агента; полный лог пишется в папку запуска

## 2. Задание

Txt-файл: секции `[task]`, `[data]`, `[automl]`, `[evaluation]`, `[agent]`, `[email]` и свободная
инструкция для LLM после `[instructions]`.

In [ ]:
TASK_FILE = 'tasks/business_type_split.txt'

task = load_task(TASK_FILE)
print(task.description)
print()
print('Инструкция для LLM:')
print(task.instructions)

## 3. Создание агента

`llm_model_type='offline'` — прогон без внешней LLM (выводы строятся формально по метрикам).
Для работы с LLM уберите параметр — будет использован `config.llm_model_type`.

In [ ]:
agent = MLAgent(TASK_FILE, config=config, project_dir=PROJECT_DIR,
                # llm_model_type='offline',
                )

# параметры задания можно поправить прямо здесь, не меняя txt:
# agent.task.data['min_rows'] = 50000
# agent.task.data['segment_values'] = 'НОВЫЙ, ПРОЛОНГАЦИЯ'
# agent.task.automl['hp_tune'] = True

## 4. Проверка без обучения (dry-run)

Загружает датасет, показывает план сегментов и колонки конфига, которых нет в данных
(агент удалит их из копии конфига перед запуском).

In [ ]:
plan = agent.dry_run()
print('Строк:', plan['rows'], '| модели:', plan['models'], '| признаков:', plan['n_features'])
print('Нет в данных:', plan['missing_columns'] or 'нет')
plan['segments']

## 5. Запуск

`send_mail=False` — сначала посмотреть отчёт, письмо отправить отдельно (п. 7).
По умолчанию берётся `[email] send` из задания.

In [ ]:
report = agent.run(send_mail=False)
print('Папка запуска:', agent.run_dir)

## 6. Результаты

In [ ]:
report  # HTML-отчёт (то же, что уйдёт в письмо)

In [ ]:
print('Рекомендация:', report.recommendation)
report.comparison

In [ ]:
report.status

In [ ]:
# Правки конфигов, которые внёс агент, и ошибки по экспериментам
for exp in report.experiments:
    print(f'— {exp.name} [{exp.status}], попыток: {exp.attempts}')
    for change in exp.config_changes:
        print('    правка:', change)
    for error in exp.errors[-2:]:
        print('    ошибка:', error.strip().splitlines()[-1][:200])

In [ ]:
import json
print(json.dumps(report.conclusions, ensure_ascii=False, indent=2))  # ответ LLM

## 7. Отправка письма

In [ ]:
agent.send()  # получатели: [email] receivers из задания или config.email_receivers
# agent.send(receivers=['me@example.ru'])

## 8. Артефакты запуска

`agent_runs/<дата>_<задание>/`:

- `report.html`, `report.json` — отчёт;
- `agent.log` — шаги агента, `llm_log.jsonl` — все запросы и ответы LLM;
- `<эксперимент>/` — `dataset.parquet`, `models_config_attemptN.json`, `models_config_final.json`,
  `attemptN.log` (полный лог mldataworker), `result.json`;
- `results/` — пиклы и артефакты mldataworker.

In [ ]:
sorted(p.name for p in agent.run_dir.iterdir())